# Laboratory 04 — The Fourier transform and convolution

A whistle has a pitch and a hand-clap does not, yet a microphone and an FFT report that the clap
contains every frequency at once. This laboratory builds the tool that makes sense of that,
measures the one inequality the rest of the course keeps meeting, and then spends its second
half on the two ways sampled data lies to you.

One step is designed to falsify: in Part 4 you will watch a tone above the Nyquist frequency
appear, cleanly and confidently, at a frequency it never had.

## Model specification

| | |
|---|---|
| **System** | square-integrable real signals and their complex spectra; numerically N samples at spacing dt |
| **Dynamics** | none — a change of representation; convolution models a linear time-invariant system when used that way |
| **Boundary** | signals decay or are windowed, and the discrete transform silently imposes periodicity on the window |
| **Ensemble** | deterministic, except the measurement cells, whose noise is Gaussian, independent and seeded |
| **Ignored** | the rigour of distribution theory — delta functions are used operationally — and convergence pathologies |
| **Valid when** | content lies below the Nyquist frequency pi/dt and the record is long against the features being resolved |
| **Failure modes** | aliasing, leakage read as physics, magnitude-only reasoning, dropping the 1/2pi or the forward sign |

All the physics lives in `wavelab.fourier`. Open it and read it — nothing here re-implements
any of it.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# wavelab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import fourier, measurement
from wavelab.validation import seed_study

DT = 0.01  # s
N = 4096
TIMES = (np.arange(N) - N // 2) * DT  # centred on t = 0, which is what spectrum() assumes
NYQUIST = np.pi / DT
BIN = 2 * np.pi / (N * DT)

rng = np.random.default_rng(2024)

print(f"record: {N} samples at dt = {DT} s, spanning {N * DT:.2f} s")
print(f"frequency bin  = {BIN:.4f} rad/s")
print(f"Nyquist        = {NYQUIST:.2f} rad/s")

## Part 1 — The pair zoo

Four transform pairs do most of the work in this course. Drawn once here, they become visual
reflexes: you should end up able to sketch any of them without integrating anything.

The numerics are dots, the closed forms are lines. Watch where they agree perfectly and where
they do not — the disagreement is as instructive as the agreement.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 8))

pairs = [
    ("rect (width 1)", fourier.rect_pulse(TIMES, 1.0),
     lambda w: fourier.sinc_spectrum(w, 1.0)),
    ("Gaussian (sigma 0.3)", fourier.gaussian_pulse(TIMES, 0.3),
     lambda w: fourier.gaussian_spectrum(w, 0.3)),
    ("one-sided decay (tau 0.5)", fourier.exp_decay(TIMES, 0.5),
     lambda w: fourier.lorentzian_spectrum(w, 0.5)),
]

for (name, signal, analytic), (left, right) in zip(pairs, axes, strict=True):
    omega, transform = fourier.spectrum(signal, DT)
    band = np.abs(omega) < 40

    left.plot(TIMES, signal, lw=1.4)
    left.set_xlim(-2, 2), left.set_xlabel("t (s)"), left.set_ylabel("f(t)")
    left.set_title(name, fontsize=9)

    right.plot(omega[band], np.abs(analytic(omega[band])), lw=2.0, color="0.7", label="closed form")
    right.plot(omega[band][::12], np.abs(transform[band][::12]), "o", markersize=2.5, label="FFT")
    right.set_xlabel("omega (rad/s)"), right.set_ylabel("|F|"), right.legend(fontsize=7)

    worst = np.max(np.abs(transform - analytic(omega)))
    print(f"{name:26s} worst |FFT - closed form| = {worst:.2e}")

plt.tight_layout()
plt.show()
print()
print("The Gaussian is smooth, so it reaches machine precision. The other two are")
print("discontinuous — the rect at both edges, the decay at t = 0 — and a sampled")
print("discontinuity costs accuracy no grid can recover. Part 8 pins the rate.")

## Part 2 — Predict

Commit to answers before running anything below.

1. A pulse is made half as long, keeping its shape. Its spectrum gets wider, narrower, or stays
   the same — by what factor?
2. Two recordings have identical |F(omega)| at every frequency. Must they be the same signal?
3. A 60 Hz sine is sampled 100 times a second. What frequency does the data show, and does
   anything in the data warn you?
4. Is there a signal both very short in time and very narrow in frequency?

**Your prediction:**

1. 
2. 
3. 
4. 

## Part 3 — The uncertainty product

RMS widths are second moments: the width of |f|^2 in time and of |F|^2 in frequency. Measure
both for a range of shapes and multiply. The claim from the page is that the product can never
fall below one half, and that exactly one shape reaches it.

The chirped Gaussian is the sharpest test in the table. Chirping does not touch |f|^2 at all, so
its duration is *identical* to the plain Gaussian's — every bit of the excess comes from phase.

In [ ]:
shapes = {
    "Gaussian, sigma = 0.2": fourier.gaussian_pulse(TIMES, 0.2),
    "Gaussian, sigma = 0.5": fourier.gaussian_pulse(TIMES, 0.5),
    "Gaussian, sigma = 1.0": fourier.gaussian_pulse(TIMES, 1.0),
    "chirped Gaussian": fourier.gaussian_pulse(TIMES, 0.5) * np.exp(1j * 3.0 * TIMES**2),
    "cosine-modulated Gaussian": fourier.gaussian_pulse(TIMES, 0.5) * np.cos(8.0 * TIMES),
    "two separated Gaussians": (fourier.gaussian_pulse(TIMES - 1.0, 0.2)
                                + fourier.gaussian_pulse(TIMES + 1.0, 0.2)),
}

print(f"{'shape':28s} {'dt_rms':>8s} {'dw_rms':>9s} {'product':>9s}")
products = {}
for name, signal in shapes.items():
    duration, bandwidth = fourier.rms_widths(signal, DT)
    products[name] = duration * bandwidth
    print(f"{name:28s} {duration:8.4f} {bandwidth:9.4f} {duration * bandwidth:9.5f}")

plt.figure(figsize=(8, 3.6))
plt.bar(range(len(products)), list(products.values()))
plt.axhline(0.5, color="green", ls="--", lw=1.2)
plt.xticks(range(len(products)), [n.split(",")[0] for n in products], rotation=20, fontsize=7)
plt.ylabel("dt_rms x dw_rms")
plt.tight_layout()
plt.show()

print()
print("Every Gaussian sits on 0.5 regardless of its width — the product is scale-free.")
print("Nothing sits below it. The chirp has the same duration as its unchirped twin.")

## Part 4 — Aliasing, measured

Prediction 3, settled. Sample a tone whose frequency is swept steadily upward, and each time ask
the spectrum what frequency it thinks it sees. Below Nyquist the answer is the truth. Above it
the answer folds back down — and looks exactly as convincing.

This is the falsifying experiment: the recorded data contain a clean peak at a frequency the
signal never had, with nothing to mark it as false.

In [ ]:
sample_dt = 0.05  # a deliberately coarse grid: Nyquist is now pi/0.05 = 62.8 rad/s
coarse_nyquist = np.pi / sample_dt
coarse_times = np.arange(2048) * sample_dt


def apparent_frequency(true_omega):
    """What the spectrum of a sampled tone reports as its frequency."""
    omega, transform = fourier.spectrum(np.cos(true_omega * coarse_times), sample_dt)
    positive = omega >= 0
    return float(omega[positive][np.argmax(np.abs(transform[positive]))])


true_rates = np.linspace(2.0, 2.9 * coarse_nyquist, 120)
apparent = np.array([apparent_frequency(w) for w in true_rates])

plt.figure(figsize=(8, 4.2))
plt.plot(true_rates, apparent, "o", markersize=3, label="measured")
plt.plot(true_rates, true_rates, lw=0.9, color="0.75", ls=":", label="the truth")
plt.axvline(coarse_nyquist, color="green", ls="--", lw=1.0, label="Nyquist")
plt.xlabel("true frequency (rad/s)"), plt.ylabel("apparent frequency (rad/s)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

for factor in (0.6, 1.4, 2.4):
    true = factor * coarse_nyquist
    print(f"true {true:7.2f} rad/s  ->  reported {apparent_frequency(true):7.2f} rad/s")
print()
print("A 60 Hz tone sampled at 100 Hz reports", end=" ")
hz_times = np.arange(2048) / 100.0
hz_omega, hz_spec = fourier.spectrum(np.cos(2 * np.pi * 60.0 * hz_times), 1 / 100.0)
hz_pos = hz_omega >= 0
print(f"{hz_omega[hz_pos][np.argmax(np.abs(hz_spec[hz_pos]))] / (2 * np.pi):.1f} Hz.")

## Part 5 — Leakage, and what a window costs

A finite record multiplies the signal by a rectangle, so its spectrum is convolved with a sinc.
A pure tone therefore does not appear as a spike — it appears as a sinc, with sidelobes spilling
across the whole axis.

How bad it looks depends on something with no physical meaning whatsoever: whether the tone's
frequency happens to land on a grid frequency of your record.

In [ ]:
on_bin = 40 * BIN  # exactly on a grid frequency
off_bin = 40.5 * BIN  # half a bin away — the worst case

hann = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N) / N))

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for ax, window, label in [(left, np.ones(N), "rectangular window"),
                          (right, hann, "Hann window")]:
    for omega_true, name in [(on_bin, "on a bin"), (off_bin, "half a bin off")]:
        omega, transform = fourier.spectrum(np.cos(omega_true * TIMES) * window, DT)
        band = (omega > on_bin - 12 * BIN) & (omega < on_bin + 12 * BIN)
        magnitude = np.abs(transform[band])
        ax.semilogy(omega[band] / BIN, magnitude / magnitude.max(), lw=1.2, label=name)
    ax.set_ylim(1e-6, 2)
    ax.set_xlabel("frequency (bins)"), ax.set_ylabel("|F| (normalised)")
    ax.set_title(label, fontsize=9), ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for window, label in [(np.ones(N), "rectangular"), (hann, "Hann")]:
    omega, transform = fourier.spectrum(np.cos(off_bin * TIMES) * window, DT)
    band = (omega > on_bin - 12 * BIN) & (omega < on_bin + 12 * BIN)
    magnitude = np.abs(transform[band]) / np.max(np.abs(transform[band]))
    far = np.abs(omega[band] - off_bin) > 4 * BIN
    print(f"{label:12s} worst sidelobe more than 4 bins out: {magnitude[far].max():.2e}")
print()
print("Hann buys four orders of magnitude of sidelobe at the cost of a main lobe")
print("about twice as wide. That trade is the whole subject of window design.")

## Part 6 — The convolution theorem, checked

Convolving in time should multiply the spectra. Two routes to the same signal: convolve
directly, or multiply the spectra and transform back. If the theorem is right — and if
`convolve` is normalised the way the page claims — the two agree to roundoff.

In [ ]:
pulse = fourier.rect_pulse(TIMES, 0.6)
smoother = fourier.gaussian_pulse(TIMES, 0.12)

direct = fourier.convolve(pulse, smoother, DT)
omega, pulse_spectrum = fourier.spectrum(pulse, DT)
_, smoother_spectrum = fourier.spectrum(smoother, DT)
via_spectra = fourier.inverse_spectrum(pulse_spectrum * smoother_spectrum, DT)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
left.plot(TIMES, pulse, lw=1.0, color="0.7", label="rect")
left.plot(TIMES, direct / direct.max(), lw=1.5, label="rect * Gaussian (scaled)")
left.set_xlim(-1, 1), left.set_xlabel("t (s)"), left.set_ylabel("f(t)"), left.legend(fontsize=8)

right.semilogy(TIMES, np.abs(direct - via_spectra.real) + 1e-20, lw=0.9)
right.set_xlim(-1, 1), right.set_xlabel("t (s)"), right.set_ylabel("|difference|")
right.set_title("the two routes, differenced", fontsize=9)
plt.tight_layout()
plt.show()

_, convolved_spectrum = fourier.spectrum(direct, DT)
relative = (np.max(np.abs(convolved_spectrum - pulse_spectrum * smoother_spectrum))
            / np.max(np.abs(pulse_spectrum * smoother_spectrum)))
print(f"spectrum(f * g) against F x G: relative difference {relative:.2e}")
print("The sharp corners of the rect have been smeared off — that is what a")
print("convolution does, and in the frequency domain it is one multiplication.")

## Part 7 — Measurement: can you resolve two tones?

Two tones, close together, buried in noise. Whether you can separate them is not a question
about the noise — it is a question about how long you recorded. The resolution of a spectrum is
roughly one bin, 2 pi / T, so the record must be long enough that the tones are more than a bin
apart.

Measure both frequencies, with uncertainties — and then look carefully at whether the
uncertainty you computed is the one that actually limits you.

In [ ]:
OMEGA_A, OMEGA_B = 20.0, 20.6  # rad/s, separated by 0.6
NOISE = 0.35
SEEDS = 16


def two_tone(n_samples, generator):
    times = (np.arange(n_samples) - n_samples // 2) * DT
    clean = np.cos(OMEGA_A * times) + np.cos(OMEGA_B * times)
    return measurement.add_noise(clean, NOISE, generator)


def spectrum_of(record):
    window = 0.5 * (1 - np.cos(2 * np.pi * np.arange(record.size) / record.size))
    return fourier.spectrum(record * window, DT)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, n_samples in zip(axes, (512, 4096), strict=True):
    omega, transform = spectrum_of(two_tone(n_samples, np.random.default_rng(7)))
    band = (omega > 18) & (omega < 23)
    ax.plot(omega[band], np.abs(transform[band]), lw=1.2)
    for line in (OMEGA_A, OMEGA_B):
        ax.axvline(line, color="0.8", ls=":", lw=0.9)
    resolution = 2 * np.pi / (n_samples * DT)
    ax.set_title(f"N = {n_samples}, one bin = {resolution:.3f} rad/s", fontsize=9)
    ax.set_xlabel("omega (rad/s)"), ax.set_ylabel("|F|")
plt.tight_layout()
plt.show()

print(f"tones are separated by {OMEGA_B - OMEGA_A:.2f} rad/s")
for n_samples in (512, 4096):
    print(f"  N = {n_samples:5d}: one bin = {2 * np.pi / (n_samples * DT):.3f} rad/s -> "
          f"{(OMEGA_B - OMEGA_A) / (2 * np.pi / (n_samples * DT)):.2f} bins apart")

In [ ]:
def refined_peak(omega, magnitude, index):
    """Sub-bin peak position by fitting a parabola to the peak and its two neighbours.

    Without this the answer is quantised to the bin grid, every seed returns the same number,
    and the reported uncertainty would be exactly zero — an error bar that lies.
    """
    y_low, y_mid, y_high = magnitude[index - 1:index + 2]
    denominator = y_low - 2 * y_mid + y_high
    shift = 0.5 * (y_low - y_high) / denominator if denominator != 0 else 0.0
    return float(omega[index] + shift * (omega[1] - omega[0]))


def measure_tone(centre, half_width=0.3):
    def measure(generator):
        omega, transform = spectrum_of(two_tone(4096, generator))
        magnitude = np.abs(transform)
        band = np.where((omega > centre - half_width) & (omega < centre + half_width))[0]
        peak = band[np.argmax(magnitude[band])]
        return refined_peak(omega, magnitude, peak)

    return measure


print("tone    measured (rad/s)      offset     offset in bins   offset in sigma")
for truth in (OMEGA_A, OMEGA_B):
    study = seed_study(measure_tone(truth), n_seeds=SEEDS)
    offset = study.mean - truth
    print(f"{truth:5.1f}   {study.mean:.4f} +/- {study.standard_error:.4f}   {offset:+.4f}   "
          f"{offset / BIN:+8.3f}      {abs(offset) / study.standard_error:8.1f}")

# The same measurement with the noise switched off, which is the whole point of the cell below.
clean_times = (np.arange(4096) - 2048) * DT
clean_record = np.cos(OMEGA_A * clean_times) + np.cos(OMEGA_B * clean_times)
omega, transform = spectrum_of(clean_record)
magnitude = np.abs(transform)
print()
for truth in (OMEGA_A, OMEGA_B):
    band = np.where((omega > truth - 0.3) & (omega < truth + 0.3))[0]
    peak = refined_peak(omega, magnitude, band[np.argmax(magnitude[band])])
    print(f"with no noise at all, the peak near {truth} sits at {peak:.4f}")

## Part 8 — Automated checks

Each assertion mirrors a claim on the module page, and each is also a test in the project's
suite. If one fails, the page is wrong — not the notebook.

In [ ]:
# 1. The smooth pair is exact; the discontinuous one is not, and its error is first order in dt.
gauss = fourier.gaussian_pulse(TIMES, 0.3)
omega, gauss_spectrum = fourier.spectrum(gauss, DT)
assert np.max(np.abs(gauss_spectrum - fourier.gaussian_spectrum(omega, 0.3))) < 1e-10

rect_errors = []
for step in (0.02, 0.01, 0.005):
    count = int(round(40.0 / step))
    grid = (np.arange(count) - count // 2) * step
    w, transform = fourier.spectrum(fourier.rect_pulse(grid, 1.0), step)
    rect_errors.append(float(np.max(np.abs(transform - fourier.sinc_spectrum(w, 1.0)))))
assert rect_errors[0] / rect_errors[-1] > 3.0, rect_errors  # halving dt twice halves it twice

# 2. Round trip and Parseval.
assert np.max(np.abs(fourier.inverse_spectrum(gauss_spectrum, DT) - gauss)) < 1e-12
time_energy = np.sum(np.abs(gauss) ** 2) * DT
spectral_energy = np.sum(np.abs(gauss_spectrum) ** 2) * BIN / (2 * np.pi)
assert np.isclose(time_energy, spectral_energy, rtol=1e-10)

# 3. The phasor sits at NEGATIVE omega0 — exactly, and only when the tone is on a bin.
phasor = 2.0 * np.exp(-1j * 0.7)
tone = np.real(phasor * np.exp(-1j * (130 * BIN) * TIMES))
w, tone_spectrum = fourier.spectrum(tone, DT)
negative = int(np.argmin(np.abs(w + 130 * BIN)))
assert np.isclose(tone_spectrum[negative] * BIN / np.pi, phasor, rtol=1e-9)

# 4. Nothing beats the bandwidth floor, and the Gaussian reaches it.
assert np.isclose(products["Gaussian, sigma = 0.5"], 0.5, rtol=1e-9)
assert all(value > 0.5 - 1e-9 for value in products.values())
assert products["chirped Gaussian"] > products["Gaussian, sigma = 0.5"]

# 5. Aliasing: a tone above Nyquist is reported, confidently, at the folded frequency.
above = 1.4 * coarse_nyquist
assert not np.isclose(apparent_frequency(above), above, rtol=0.02)
assert np.isclose(apparent_frequency(above), 2 * coarse_nyquist - above, rtol=0.02)

# 6. The convolution theorem.
assert relative < 1e-10, relative

# 7. Both tones recovered to a fraction of a bin — and note what is NOT asserted here.
#    The seed study's error bar describes the noise alone, and the noise is not what limits
#    this measurement: each peak is pulled toward its neighbour by the tail of the other tone's
#    main lobe, a systematic offset of a few hundredths of a bin that survives with the noise
#    switched off entirely. It is twenty to forty times the statistical scatter, so a
#    three-sigma test against the truth would fail on a measurement that is in fact good to
#    0.04%. The budget below is a total error budget, in the units the resolution is set by.
for truth in (OMEGA_A, OMEGA_B):
    study = seed_study(measure_tone(truth), n_seeds=SEEDS)
    assert abs(study.mean - truth) < 0.1 * BIN, (truth, study.mean)
    assert study.standard_error < 0.02 * BIN  # the noise really is the smaller problem

print("all checks passed")
print(f"bandwidth floor: a {2.0} ms pulse cannot be narrower than {0.5 / 2.0e-3:.0f} rad/s")

## Part 9 — Explore it yourself

The panel below redraws on a **Run** button (`interact_manual`) rather than on every slider
twitch: the browser kernel is too slow for that.

Three things worth trying:

1. Chirp a Gaussian and watch the product meter leave one half while the pulse shape on the left
   does not change at all. Where did the extra bandwidth come from?
2. Widen the pulse until its spectrum is narrower than a few bins, and see the measurement
   degrade — you cannot measure a bandwidth finer than your record allows.
3. Turn the sampling control coarse and push the carrier up through Nyquist.

In [ ]:
import ipywidgets as widgets


def sculpt(shape="Gaussian", width=0.4, chirp=0.0, carrier=0.0):
    if shape == "Gaussian":
        signal = fourier.gaussian_pulse(TIMES, width)
    elif shape == "rect":
        signal = fourier.rect_pulse(TIMES, 4 * width)
    else:
        signal = fourier.exp_decay(TIMES, width)

    signal = signal * np.exp(1j * chirp * TIMES**2)
    if carrier:
        signal = signal * np.cos(carrier * TIMES)

    omega, transform = fourier.spectrum(signal, DT)
    duration, bandwidth = fourier.rms_widths(signal, DT)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
    left.plot(TIMES, np.abs(signal), lw=1.5)
    left.set_xlim(-3, 3), left.set_xlabel("t (s)"), left.set_ylabel("|f(t)|")
    band = np.abs(omega) < 60
    right.plot(omega[band], np.abs(transform[band]), lw=1.5)
    right.set_xlabel("omega (rad/s)"), right.set_ylabel("|F(omega)|")
    plt.tight_layout()
    plt.show()

    print(f"dt_rms = {duration:.4f} s   dw_rms = {bandwidth:.4f} rad/s   "
          f"product = {duration * bandwidth:.4f}   (floor is 0.5)")


widgets.interact_manual(
    sculpt,
    shape=widgets.Dropdown(options=["Gaussian", "rect", "one-sided decay"], value="Gaussian"),
    width=widgets.FloatSlider(min=0.05, max=1.5, step=0.05, value=0.4, description="width"),
    chirp=widgets.FloatSlider(min=0.0, max=8.0, step=0.25, value=0.0, description="chirp"),
    carrier=widgets.FloatSlider(min=0.0, max=40.0, step=1.0, value=0.0, description="carrier"),
);

## Check your understanding

Six questions on the module's objectives.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "04-fourier-transform.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

1. "What frequency is a hand-clap?" — say why the question is ill-posed and what replaces it.
   Your answer should name what changed between module 03 and this one.
2. Your Part 7 error bars: which limits you more, the noise or the record length? Say how you
   would tell the two apart using only the seeds.
3. A colleague discards the negative-frequency half of every FFT "because it is redundant". When
   are they safe, and what exactly have they thrown away when they are not?

**Your answers:**

1. 
2. 
3. 